# Multimodal Retrieval Augmented Generation with Content Understanding

# Overview

Azure AI Content Understanding provides a powerful solution for extracting data from diverse content types, while preserving semantic integrity and contextual relationships ensuring optimal performance in Retrieval Augmented Generation (RAG) applications.

This sample demonstrates how to leverage Azure AI's Content Understanding capabilities to extract:

- OCR and layout information from documents
- Image description, summarization and classification
- Audio transcription with speaker diarization from audio files
- Shot detection, keyframe extraction, and audio transcription from videos

This notebook illustrates how to extract content from unstructured multimodal data and apply it to Retrieval Augmented Generation (RAG). The resulting output can be converted to vector embeddings and indexed in Azure AI Search. When a user submits a query, Azure AI Search retrieves relevant chunks to generate a context-aware response.


# Pre-requisites
1. Follow [README](../README.md#configure-azure-ai-service-resource) to create essential resource that will be used in this sample
2. Install required packages

In [ ]:
%pip install -r ../requirements.txt

# Load environment variables

In [ ]:

import os
from dotenv import load_dotenv
load_dotenv()

# Load and validate Azure AI Services configs
AZURE_AI_SERVICE_ENDPOINT = os.getenv("AZURE_AI_SERVICE_ENDPOINT")
AZURE_AI_SERVICE_API_VERSION = os.getenv("AZURE_AI_SERVICE_API_VERSION") or "2024-12-01-preview"
AZURE_DOCUMENT_INTELLIGENCE_API_VERSION = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_API_VERSION") or "2024-11-30"

# Load and validate Azure OpenAI configs
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
AZURE_OPENAI_CHAT_API_VERSION = os.getenv("AZURE_OPENAI_CHAT_API_VERSION") or "2024-08-01-preview"
AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME")
AZURE_OPENAI_EMBEDDING_API_VERSION = os.getenv("AZURE_OPENAI_EMBEDDING_API_VERSION") or "2023-05-15"

# Load and validate Azure Search Services configs
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX_NAME") or "sample-doc-index"

# Create custom analyzer

In [ ]:
from langchain_classic import hub
from langchain_openai import AzureChatOpenAI
from langchain_openai import AzureOpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.passthrough import RunnablePassthrough
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.vectorstores.azuresearch import AzureSearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents.base import Document
import requests
import json
import sys
import uuid
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Add the parent directory to the path to use shared modules
parent_dir = Path(Path.cwd()).parent
sys.path.append(str(parent_dir))


### Create analyzers with pre-defined schemas.
Feel free to start with the provided sample data as a reference and experiment with your own data to explore its capabilities.

In [ ]:
from pathlib import Path
from python.content_understanding_client import AzureContentUnderstandingClient
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
analyzer_configs = []
# Specify the folder path
folder_path = Path("../data")

# List all files in the folder
files = [file for file in folder_path.iterdir() if file.is_file()]

#Assign docs to the analyzers
for file in files:
    print(file.name.lower())
    if "application" in file.name.lower():
        config = {
            "id": "application-analyzer",
            "template_path": "../analyzer_templates/schema_application.json",
            "location": file,
        }
    elif "statement" in file.name.lower():
        config = {
            "id": "statement-analyzer",
            "template_path": "../analyzer_templates/schema_bankstmt.json",
            "location": file,
        }
    elif "ssn" in file.name.lower():
        config = {
            "id": "ssn-analyzer",
            "template_path": "../analyzer_templates/SSN_image.json",
            "location": file,
        }
    elif ".pdf" in file.suffix.lower():
        config = {
            "id": "pdf-analyzer",
            "template_path": "../analyzer_templates/content_document.json",
            "location": file,
        }
    else: 
        print(f"We do not currently handle {file.suffix} file types that are not a statement, application, or ssn.")
        print("------------------")

        continue
    print(f"Assigned {file.name} to analyzer {config['id']}")
    print("------------------")

    analyzer_configs.append(config)


# Create Content Understanding client
content_understanding_client = AzureContentUnderstandingClient(
    endpoint=AZURE_AI_SERVICE_ENDPOINT,
    api_version=AZURE_AI_SERVICE_API_VERSION,
    token_provider=token_provider,
    x_ms_useragent="azure-ai-content-understanding-python/content_extraction", # This header is used for sample usage telemetry, please comment out this line if you want to opt out.
)

# Iterate through each config and create an analyzer
for analyzer in analyzer_configs:
    analyzer_id = analyzer["id"]
    template_path = analyzer["template_path"]

    try: 
        try:
            content_understanding_client.get_analyzer_detail_by_id(analyzer_id)
            print(f"Analyzer already exists: {analyzer_id}, skipping creation.")
            print("------------------")

            continue
        except Exception as e:
            # Create the analyzer using the content understanding client
            response = content_understanding_client.begin_create_analyzer(
                analyzer_id=analyzer_id,
                analyzer_template_path=template_path
            )
            result = content_understanding_client.poll_result(response)
            print(f"Successfully created analyzer: {analyzer_id}")
            print("------------------")
    except Exception as e:
            print(f"Failed to create analyzer: {analyzer_id}")
            print(f"Error: {e}")

### Use created analyzers to extract multimodal content

In [ ]:

#Iterate through each analyzer created and analyze content for each modality

analyzer_results =[]
extracted_markdown = []
analyzer_content = []
for analyzer in analyzer_configs:
    analyzer_id = analyzer["id"]
    template_path = analyzer["template_path"]
    file_location = analyzer["location"]
    try:
           # Analyze content
            response = content_understanding_client.begin_analyze(analyzer_id, file_location)
            result = content_understanding_client.poll_result(response)
            analyzer_results.append({"id":analyzer_id, "result": result["result"]})
            analyzer_content.append({"id": analyzer_id, "content": result["result"]["contents"]})
                       
    except Exception as e:
            print(e)
            print("Error in creating analyzer. Please double-check your analysis settings.\nIf there is a conflict, you can delete the analyzer and then recreate it, or move to the next cell and use the existing analyzer.")
            print("------------------")

print("Analyzer Results:")
for analyzer_result in analyzer_results:
    print(f"Analyzer ID: {analyzer_result['id']}")
    print(json.dumps(analyzer_result["result"], indent=2))
    print("------------------")
            

# print("Analyzer Content:")
# for content in analyzer_content:
#     print(f"Content ID: {content['id']}")
#     print(json.dumps(content["content"], indent=2))   
# Delete the analyzer if it is no longer needed
#content_understanding_client.delete_analyzer(ANALYZER_ID)

# Organize multimodal data
This is a simple starting point. Feel free to give your own chunking strategies a try!

### Preprocess JSON output data

In [ ]:
    # Configure langchain text splitting settings
from langchain_text_splitters import CharacterTextSplitter


EMBEDDING_CHUNK_SIZE = 512
EMBEDDING_CHUNK_OVERLAP = 10
# Split the document into chunks base on markdown headers.
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"),
    ("#####", "Header 5"),
    ("######", "Header 6"),
    ("#######", "Header 7"),
    ("########", "Header 8"),
    ("#########", "Header 9"),
]
def convert_values_to_strings(json_obj):
    return [str(value) for value in json_obj]

#process all content and convert to string      
def process_allJSON_content(all_content):
    print(all_content)
    output = []
    i=0
    for content in all_content:
        print(f"Processing element : " + str(i) + " content ID: " + str(content['id']))
        if "pdf-analyzer" in content['id']:
            text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
            # text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
            #     encoding_name="cl100k_base", chunk_size=EMBEDDING_CHUNK_SIZE, chunk_overlap=EMBEDDING_CHUNK_OVERLAP
            # )
            docs_string = analyzer_content[i]['content'][0]['markdown']
            docs_splits1 = text_splitter.split_text(docs_string)
            # Wrap the strings as Document objects for consistency
            #docs_splits1 = [Document(page_content=v) for v in docs_splits1]
            # for doc in docs_splits1:
            #     print(f"doc content", doc.page_content)    
            output += docs_splits1

            #check output after processing document analyzer content
            print("After processing pdf analyzer content: " + str(i)+", there are " + str(len(output)) + " documents.")
        else:
        # Initialize empty list to store string of all content
            document_splits = [
                "This is a json string representing a document with text and metadata for the file located in "+str(analyzer_configs[i]["location"])+" "
                + v 
                + "```"
                for v in convert_values_to_strings(all_content[i]["content"])
            ]
            docs = [Document(page_content=v) for v in document_splits]
            output += docs
        i+=1
    
    return output

all_splits = process_allJSON_content(analyzer_content)

print("There are " + str(len(all_splits)) + " documents.") 
# Print the content of all doc splits
# for doc in all_splits:
#     print(f"doc content", doc.page_content)

# Embed and index the chunks

In [ ]:
# Embed the splitted documents and insert into Azure Search vector store
def embed_and_index_chunks(docs):
    aoai_embeddings = AzureOpenAIEmbeddings(
        azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME,
        openai_api_version=AZURE_OPENAI_EMBEDDING_API_VERSION,  # e.g., "2023-12-01-preview"
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        azure_ad_token_provider=token_provider
    )

    vector_store: AzureSearch = AzureSearch(
        azure_search_endpoint=AZURE_SEARCH_ENDPOINT,
        azure_search_key=None,
        index_name=AZURE_SEARCH_INDEX_NAME,
        embedding_function=aoai_embeddings.embed_query
    )
    vector_store.add_documents(documents=docs)
    return vector_store


# embed and index the docs:
vector_store = embed_and_index_chunks(all_splits)